In [50]:
import torch
import torch.nn as nn

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

DATA PREPARATION!

In [51]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# dataset
df = pd.read_csv("fmnist_small.csv")

# split train and test
X = df.iloc[:,1:]
y = df.iloc[:,0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=24)

# Scale only the pixels not the labels!
X_train_scaled = X_train/255.0
X_test_scaled = X_test/255.0

# Convert all to tensors
X_train_tensor = torch.tensor(X_train_scaled.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test.values, dtype=torch.long)

DATA LOADING!

In [52]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, X_train_tensor, y_train_tensor):
        self.X_train_tensor = X_train_tensor
        self.y_train_tensor = y_train_tensor
    def __len__(self):
        return len(self.X_train_tensor)
    def __getitem__(self,idx):
        return self.X_train_tensor[idx], self.y_train_tensor[idx]

train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True, pin_memory=True)

MANUAL BUILDING OF THE CNN MODEL ARCHITECTURE!

In [ ]:
class MyNN(nn.Module):
  def __init__(self, num_channels):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(num_channels, 128, kernel_size=3, padding="same"),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(128, 64, kernel_size=3, padding="same"),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),

        nn.Linear(64*7*7, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p=0.3),

        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p=0.3),

        nn.Linear(64, 10)
    )
  def forward(self, input):
    image = input.reshape(-1,1,28,28)
    output = self.features(image)
    output = self.classifier(output)
    return output

In [54]:
from torch import optim

# set learning rate and epochs
epochs = 50
learning_rate = 0.1
lambda_val = 1e-4

# instatiate the model: 1 --> number of chennels
num_channels = 1
model = MyNN(num_channels).to(device)
# loss function
loss_fun = nn.CrossEntropyLoss()
# optimizer
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=lambda_val)

TRAINING LOOP!

In [55]:
# training loop
for epoch in range(epochs):
  total_epoch_loss = 0
  for batch_features, batch_labels in train_loader:
    # zero the prev grads
    optimizer.zero_grad()
    # move the tensor data to GPU!
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    # forward pass
    outputs = model.forward(batch_features)
    # calculate loss
    loss = loss_fun(outputs, batch_labels)
    # back pass
    loss.backward()
    # update grads
    optimizer.step()
    # calculate loss for all items in a batch
    total_epoch_loss = total_epoch_loss + loss.item()
  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')


Epoch: 1 , Loss: 0.8890808200836182
Epoch: 2 , Loss: 0.5652695832649867
Epoch: 3 , Loss: 0.4710871557891369
Epoch: 4 , Loss: 0.4344642659028371
Epoch: 5 , Loss: 0.401255690852801
Epoch: 6 , Loss: 0.3710229856769244
Epoch: 7 , Loss: 0.3400491100549698
Epoch: 8 , Loss: 0.3026017523060242
Epoch: 9 , Loss: 0.2876178174217542
Epoch: 10 , Loss: 0.27163520162304244
Epoch: 11 , Loss: 0.26268931542833646
Epoch: 12 , Loss: 0.24976410726706186
Epoch: 13 , Loss: 0.23571173583467803
Epoch: 14 , Loss: 0.2195318488528331
Epoch: 15 , Loss: 0.21349838249385356
Epoch: 16 , Loss: 0.19870310944815478
Epoch: 17 , Loss: 0.17745991138120493
Epoch: 18 , Loss: 0.18828408161799112
Epoch: 19 , Loss: 0.18415068330864112
Epoch: 20 , Loss: 0.16220315589259068
Epoch: 21 , Loss: 0.16410758465528488
Epoch: 22 , Loss: 0.15145707607269288
Epoch: 23 , Loss: 0.1522945956637462
Epoch: 24 , Loss: 0.136924502539138
Epoch: 25 , Loss: 0.1410223183905085
Epoch: 26 , Loss: 0.13092144288122654
Epoch: 27 , Loss: 0.1238063480084141

EVALUATION LOOP ON TEST DATA!

In [56]:
# set model to eval mode
model.eval()

# evaluation code
total = len(y_test_tensor)
correct = 0
with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    outputs = model(batch_features)
    logit_values, predicted = torch.max(outputs, 1)
    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.8833333333333333


EVALUATION LOOP ON TRAIN DATA!

In [57]:
# set model to eval mode
model.eval()

# evaluation code
total = len(y_train_tensor)
correct = 0
with torch.no_grad():
  for batch_features, batch_labels in train_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    outputs = model(batch_features)
    logit_values, predicted = torch.max(outputs, 1)
    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.99625
